# 05. TTL Data Augmentation Defense

This notebook explores data augmentation as a defense against TTL perturbations: evaluates augmentation strength sweeps (+/-2%, +/-5%, +/-10%, +/-15%), reproducible MLP augmentation experiments across 3 seeds, and produces Table 8 comparing Standard vs. TTL-augmented MLP under dTTL perturbations.


MLP+DATA AUGMENTATION


In [33]:
# ============================================================
# TTL AUGMENTATION-STRENGTH SWEEP
# MLP BASELINE vs TTL-AUGMENTED MLP
#

# UNSW-NB15
#
# IMPORTANT:
# - 0% augmentation = STANDARD CLEAN MLP
# - 50% of TRAINING samples are augmented
# - Both sttl and dttl are perturbed
# - Test set is NEVER augmented
# - Augmentation levels: ±2%, ±5%, ±10%, ±15%
# - Three independent seeds: 42, 123, 2026
# ============================================================

import numpy as np
import pandas as pd
import random

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

TARGET = "label"

DROP_COLS = [
    "label",
    "attack_cat"
]

# Three seeds used throughout the experiments
SEEDS = [42, 123, 2026]

# Augmentation strengths
AUGMENTATION_LEVELS = [
    0,
    2,
    5,
    10,
    15
]

# IMPORTANT:
# 50% of training samples are selected for augmentation
AUGMENTATION_FRACTION = 0.50

EPOCHS = 10
BATCH_SIZE = 512
LEARNING_RATE = 1e-3


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic behaviour where possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. DATA
# ============================================================

feature_cols = [
    c for c in train_df.columns
    if c not in DROP_COLS
]

X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

y_train = train_df[TARGET].astype(int).to_numpy()
y_test = test_df[TARGET].astype(int).to_numpy()


print("=" * 80)
print("TTL AUGMENTATION EXPERIMENT")
print("=" * 80)

print("Training samples :", len(X_train_raw))
print("Testing samples  :", len(X_test_raw))

print("\nAugmentation fraction:", AUGMENTATION_FRACTION)

print("Seeds:", SEEDS)

print("Augmentation levels:", AUGMENTATION_LEVELS)

print("\nTTL columns:")
print("sttl")
print("dttl")


# ============================================================
# 4. CHECK TTL FEATURES
# ============================================================

for col in ["sttl", "dttl"]:

    if col not in X_train_raw.columns:
        raise ValueError(
            f"Required TTL feature '{col}' was not found "
            f"in the training dataframe."
        )


# ============================================================
# 5. PREPROCESSING
# ============================================================

categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()


numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_cols
    )
])


print("\nFitting preprocessing pipeline...")

X_train_processed = preprocessor.fit_transform(
    X_train_raw
)

X_test_processed = preprocessor.transform(
    X_test_raw
)


X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)


print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)


# ============================================================
# 6. FIND TTL COLUMN POSITIONS AFTER PREPROCESSING
# ============================================================
#
# sttl and dttl are numeric features.
# ColumnTransformer keeps numeric features first, so their
# positions are obtained from numeric_cols.
# ============================================================

sttl_idx = numeric_cols.index("sttl")
dttl_idx = numeric_cols.index("dttl")

print("\nProcessed TTL column positions:")
print("sttl index:", sttl_idx)
print("dttl index:", dttl_idx)


# ============================================================
# 7. MLP ARCHITECTURE
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x).squeeze(1)


# ============================================================
# 8. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 9. TTL AUGMENTATION FUNCTION
# ============================================================
#
# EXACT EXPERIMENT:
#
# For each augmentation level:
#
#   ±2%
#   ±5%
#   ±10%
#   ±15%
#
# 50% of the TRAINING samples are selected.
#
# For the selected samples:
#
#   sttl = sttl * (1 + random perturbation)
#   dttl = dttl * (1 + random perturbation)
#
# where the perturbation is uniformly sampled between
# -level and +level.
#
# Example for ±10%:
#
#   multiplier ∈ [0.90, 1.10]
#
# The remaining 50% are unchanged.
#
# TEST DATA IS NEVER MODIFIED.
# ============================================================

def augment_ttl_features(
    X,
    strength_percent,
    fraction,
    seed,
    sttl_index,
    dttl_index
):

    X_aug = X.copy()

    # --------------------------------------------------------
    # 0% = completely clean training data
    # --------------------------------------------------------

    if strength_percent == 0:

        return X_aug


    rng = np.random.default_rng(seed)

    n_samples = len(X_aug)

    n_selected = int(
        np.floor(
            fraction * n_samples
        )
    )

    # --------------------------------------------------------
    # Select exactly 50% of training samples
    # --------------------------------------------------------

    selected_indices = rng.choice(
        n_samples,
        size=n_selected,
        replace=False
    )


    # --------------------------------------------------------
    # Random perturbation independently for each sample
    # and each TTL feature.
    #
    # Example:
    # strength = 10
    #
    # perturbation ∈ [-0.10, +0.10]
    # --------------------------------------------------------

    max_change = strength_percent / 100.0


    sttl_perturbation = rng.uniform(
        -max_change,
        max_change,
        size=n_selected
    )

    dttl_perturbation = rng.uniform(
        -max_change,
        max_change,
        size=n_selected
    )


    # --------------------------------------------------------
    # Apply multiplicative perturbations
    # --------------------------------------------------------

    X_aug[
        selected_indices,
        sttl_index
    ] *= (
        1.0 + sttl_perturbation
    )


    X_aug[
        selected_indices,
        dttl_index
    ] *= (
        1.0 + dttl_perturbation
    )


    return X_aug


# ============================================================
# 10. TRAIN + EVALUATE MLP
# ============================================================

def train_and_evaluate_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    seed
):

    set_seed(seed)

    # --------------------------------------------------------
    # Convert to tensors
    # --------------------------------------------------------

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train.astype(np.float32),
        dtype=torch.float32
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )


    # IMPORTANT:
    # DataLoader shuffle is seeded because set_seed(seed)
    # was called immediately before creating it.
    # --------------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = MLP(
        input_dim=X_train.shape[1]
    ).to(device)


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    for epoch in range(EPOCHS):

        total_loss = 0.0

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                * len(batch_X)
            )


    # --------------------------------------------------------
    # Test evaluation
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        logits = model(
            X_test_tensor.to(device)
        )

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()


    predictions = (
        probabilities >= 0.5
    ).astype(int)


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    tpr = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    auroc = roc_auc_score(
        y_test,
        probabilities
    )

    auprc = average_precision_score(
        y_test,
        probabilities
    )


    return tpr, auroc, auprc


# ============================================================
# 11. RUN AUGMENTATION SWEEP
# ============================================================

all_results = []


print("\n")
print("=" * 80)
print("STARTING 50% TTL AUGMENTATION SWEEP")
print("=" * 80)


for strength in AUGMENTATION_LEVELS:

    print("\n" + "-" * 80)

    if strength == 0:

        print(
            "0% AUGMENTATION "
            "(STANDARD CLEAN MLP)"
        )

    else:

        print(
            f"±{strength}% TTL AUGMENTATION "
            f"(50% TRAINING SAMPLES)"
        )


    print("-" * 80)


    for seed in SEEDS:

        print(
            f"\nSeed: {seed}"
        )


        # ----------------------------------------------------
        # Create augmented training data
        # ----------------------------------------------------

        X_train_augmented = augment_ttl_features(
            X=X_train_processed,
            strength_percent=strength,
            fraction=AUGMENTATION_FRACTION,
            seed=seed,
            sttl_index=sttl_idx,
            dttl_index=dttl_idx
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # X_test_processed is passed unchanged.
        # ----------------------------------------------------

        tpr, auroc, auprc = train_and_evaluate_mlp(
            X_train=X_train_augmented,
            y_train=y_train,
            X_test=X_test_processed,
            y_test=y_test,
            seed=seed
        )


        print(
            f"TPR   : {tpr:.6f}"
        )

        print(
            f"AUROC : {auroc:.6f}"
        )

        print(
            f"AUPRC : {auprc:.6f}"
        )


        all_results.append({

            "Augmentation_%": strength,

            "Seed": seed,

            "TPR": tpr,

            "AUROC": auroc,

            "AUPRC": auprc
        })


# ============================================================
# 12. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    all_results
)


# ============================================================
# 13. MEAN ± SD
# ============================================================

summary = (
    results_df
    .groupby("Augmentation_%")
    .agg(

        TPR_mean=(
            "TPR",
            "mean"
        ),

        TPR_SD=(
            "TPR",
            "std"
        ),

        AUROC_mean=(
            "AUROC",
            "mean"
        ),

        AUROC_SD=(
            "AUROC",
            "std"
        ),

        AUPRC_mean=(
            "AUPRC",
            "mean"
        ),

        AUPRC_SD=(
            "AUPRC",
            "std"
        )
    )
    .reset_index()
)


# ============================================================
# 14. PRINT FULL SEED RESULTS
# ============================================================

print("\n")
print("=" * 100)
print("RESULTS FOR ALL THREE SEEDS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 15. PRINT PAPER-READY SUMMARY
# ============================================================

print("\n")
print("=" * 110)
print("TTL AUGMENTATION-STRENGTH SWEEP — MEAN ± SD")
print("=" * 110)

print(
    f"{'Augmentation':<15}"
    f"{'TPR':<25}"
    f"{'AUROC':<25}"
    f"{'AUPRC':<25}"
)

print("-" * 110)


for _, row in summary.iterrows():

    strength = int(
        row["Augmentation_%"]
    )

    if strength == 0:

        label = "0%"

    else:

        label = f"±{strength}%"


    tpr_string = (
        f"{row['TPR_mean']:.4f} ± "
        f"{row['TPR_SD']:.4f}"
    )

    auroc_string = (
        f"{row['AUROC_mean']:.4f} ± "
        f"{row['AUROC_SD']:.4f}"
    )

    auprc_string = (
        f"{row['AUPRC_mean']:.4f} ± "
        f"{row['AUPRC_SD']:.4f}"
    )


    print(
        f"{label:<15}"
        f"{tpr_string:<25}"
        f"{auroc_string:<25}"
        f"{auprc_string:<25}"
    )


# ============================================================
# 16. PAPER-READY TABLE 7
# ============================================================

paper_table = summary.copy()

paper_table["Augmentation"] = (
    paper_table["Augmentation_%"]
    .apply(
        lambda x:
        "0%"
        if x == 0
        else f"±{int(x)}%"
    )
)


paper_table = paper_table[
    [
        "Augmentation",
        "TPR_mean",
        "TPR_SD",
        "AUROC_mean",
        "AUROC_SD",
        "AUPRC_mean",
        "AUPRC_SD"
    ]
]


paper_table.columns = [

    "Augmentation",

    "TPR_mean",
    "TPR_SD",

    "AUROC_mean",
    "AUROC_SD",

    "AUPRC_mean",
    "AUPRC_SD"
]


print("\n")
print("=" * 100)
print("PAPER-READY TABLE")
print("=" * 100)

print(
    paper_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 17. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "TTL_augmentation_all_seeds.csv",
    index=False
)

paper_table.to_csv(
    "TTL_augmentation_summary.csv",
    index=False
)


print("\n")
print("=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    "TTL_augmentation_all_seeds.csv"
)

print(
    "TTL_augmentation_summary.csv"
)


# ============================================================
# 18. IMPORTANT EXPERIMENT CHECKS
# ============================================================

print("\n")
print("=" * 80)
print("EXPERIMENT CHECKS")
print("=" * 80)

print(
    "✓ 0% augmentation = standard clean MLP"
)

print(
    "✓ 50% of training samples selected for augmentation"
)

print(
    "✓ Both sttl and dttl perturbed"
)

print(
    "✓ Augmentation applied ONLY to training data"
)

print(
    "✓ Test data remained completely unchanged"
)

print(
    "✓ Augmentation levels = ±2%, ±5%, ±10%, ±15%"
)

print(
    "✓ Seeds = 42, 123, 2026"
)

print(
    "✓ Results reported as mean ± SD"
)

print(
    "✓ attack_cat excluded to prevent label leakage"
)

TTL AUGMENTATION EXPERIMENT
Training samples : 175341
Testing samples  : 82332

Augmentation fraction: 0.5
Seeds: [42, 123, 2026]
Augmentation levels: [0, 2, 5, 10, 15]

TTL columns:
sttl
dttl

Fitting preprocessing pipeline...
Processed train shape: (175341, 195)
Processed test shape : (82332, 195)

Processed TTL column positions:
sttl index: 7
dttl index: 8

Device: cuda


STARTING 50% TTL AUGMENTATION SWEEP

--------------------------------------------------------------------------------
0% AUGMENTATION (STANDARD CLEAN MLP)
--------------------------------------------------------------------------------

Seed: 42
TPR   : 0.426586
AUROC : 0.677026
AUPRC : 0.755906

Seed: 123
TPR   : 0.412711
AUROC : 0.668383
AUPRC : 0.743579

Seed: 2026
TPR   : 0.430226
AUROC : 0.667216
AUPRC : 0.753543

--------------------------------------------------------------------------------
±2% TTL AUGMENTATION (50% TRAINING SAMPLES)
--------------------------------------------------------------------------

In [34]:
# ============================================================
# TTL AUGMENTATION ROBUSTNESS EXPERIMENT
# STANDARD MLP vs TTL-AUGMENTED MLP
#
# 50% OF TRAINING SAMPLES ARE AUGMENTED
# Seeds: 42, 123, 2026
# ============================================================

import numpy as np
import pandas as pd
import random

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEEDS = [42, 123, 2026]

# dTTL perturbations applied to TEST SET
DTTL_LEVELS = [-15, -10, -5, -2, 0, 2, 5, 10, 15]

# ------------------------------------------------------------
# IMPORTANT:
# 50% of training samples will be selected for augmentation
# ------------------------------------------------------------

AUGMENT_FRACTION = 0.50

# Strength of TTL augmentation during training.
#
# If your TTL-augmentation experiment is the ±15% experiment,
# keep this as 15.
#
# You can change to 2, 5, 10, or 15 for the augmentation sweep.
# ------------------------------------------------------------

AUGMENTATION_STRENGTH = 15

EPOCHS = 10
BATCH_SIZE = 512
LEARNING_RATE = 1e-3

TARGET = "label"

DROP_COLS = [
    "label",
    "attack_cat"
]


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Make CUDA operations as deterministic as possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. DATA
# ============================================================

feature_cols = [
    c for c in train_df.columns
    if c not in DROP_COLS
]

X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

y_train = train_df[TARGET].astype(int).to_numpy()
y_test = test_df[TARGET].astype(int).to_numpy()


print("=" * 80)
print("TTL AUGMENTATION ROBUSTNESS EXPERIMENT")
print("=" * 80)

print("Training samples :", len(X_train_raw))
print("Testing samples  :", len(X_test_raw))
print("Features         :", len(feature_cols))

print("\nAugmentation fraction:", AUGMENT_FRACTION)
print("Augmentation strength:", f"±{AUGMENTATION_STRENGTH}%")

print("Seeds:", SEEDS)


# ============================================================
# 4. IDENTIFY FEATURE TYPES
# ============================================================

categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()


print("\nNumeric features    :", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


# ============================================================
# 5. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_cols
    )
])


print("\nFitting preprocessing pipeline...")

X_train_processed = preprocessor.fit_transform(
    X_train_raw
)

X_test_processed = preprocessor.transform(
    X_test_raw
)


X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)


print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)


# ============================================================
# 6. FIND sttl AND dttl COLUMNS
# ============================================================
#
# We need to perturb the ORIGINAL numeric TTL values before
# preprocessing.
#
# This avoids incorrectly perturbing the standardized values.
# ============================================================

if "sttl" not in X_train_raw.columns:
    raise ValueError(
        "sttl column was not found in train_df."
    )

if "dttl" not in X_train_raw.columns:
    raise ValueError(
        "dttl column was not found in train_df."
    )


# ============================================================
# 7. TEST-SET dTTL PERTURBATION
# ============================================================

def perturb_dttl_test(
    X_test_original,
    percentage
):
    """
    Apply a controlled multiplicative dTTL perturbation.

    Example:
        -15% -> dttl * 0.85
        -10% -> dttl * 0.90
         +10% -> dttl * 1.10
    """

    X = X_test_original.copy()

    factor = 1.0 + (
        percentage / 100.0
    )

    X["dttl"] = (
        X["dttl"] * factor
    )

    return X


# ============================================================
# 8. TTL TRAINING AUGMENTATION
# ============================================================

def create_ttl_augmented_training_data(
    X_original,
    y,
    seed,
    augmentation_strength,
    fraction=0.50
):
    """
    Select exactly `fraction` of training samples and perturb
    BOTH sttl and dttl.

    The remaining training samples remain unchanged.

    Example for ±15%:
        sttl -> multiplied by a random factor between 0.85 and 1.15
        dttl -> multiplied by a random factor between 0.85 and 1.15

    The selection and perturbations are deterministic for each seed.
    """

    rng = np.random.default_rng(seed)

    X_aug = X_original.copy()

    n_samples = len(X_aug)

    n_augmented = int(
        round(
            n_samples * fraction
        )
    )

    # Select 50% of samples
    selected_indices = rng.choice(
        n_samples,
        size=n_augmented,
        replace=False
    )

    # Random perturbation factors
    lower = 1.0 - (
        augmentation_strength / 100.0
    )

    upper = 1.0 + (
        augmentation_strength / 100.0
    )

    sttl_factors = rng.uniform(
        lower,
        upper,
        size=n_augmented
    )

    dttl_factors = rng.uniform(
        lower,
        upper,
        size=n_augmented
    )

    # Apply perturbation only to selected samples
    X_aug.loc[
        X_aug.index[selected_indices],
        "sttl"
    ] = (
        X_aug.loc[
            X_aug.index[selected_indices],
            "sttl"
        ].to_numpy()
        * sttl_factors
    )

    X_aug.loc[
        X_aug.index[selected_indices],
        "dttl"
    ] = (
        X_aug.loc[
            X_aug.index[selected_indices],
            "dttl"
        ].to_numpy()
        * dttl_factors
    )

    return X_aug


# ============================================================
# 9. MLP MODEL
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        return self.network(
            x
        ).squeeze(1)


# ============================================================
# 10. TRAIN MLP
# ============================================================

def train_mlp(
    X,
    y,
    seed
):

    set_seed(seed)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    X_np = np.asarray(
        X,
        dtype=np.float32
    )

    y_np = np.asarray(
        y,
        dtype=np.float32
    )

    X_tensor = torch.tensor(
        X_np,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_np,
        dtype=torch.float32
    )

    dataset = TensorDataset(
        X_tensor,
        y_tensor
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )

    model = MLP(
        X_np.shape[1]
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()

        for batch_X, batch_y in loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(
                batch_X
            )

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

    return model, device


# ============================================================
# 11. MODEL PREDICTION
# ============================================================

def predict_mlp(
    model,
    device,
    X
):

    model.eval()

    X_np = np.asarray(
        X,
        dtype=np.float32
    )

    X_tensor = torch.tensor(
        X_np,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        logits = model(
            X_tensor
        )

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()

    return probabilities


# ============================================================
# 12. EVALUATION
# ============================================================

def calculate_metrics(
    y_true,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    tpr = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    auroc = roc_auc_score(
        y_true,
        probabilities
    )

    auprc = average_precision_score(
        y_true,
        probabilities
    )

    return (
        tpr,
        auroc,
        auprc
    )


# ============================================================
# 13. STORAGE
# ============================================================

standard_records = []
augmented_records = []


# ============================================================
# 14. RUN EXPERIMENT FOR EACH SEED
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)


    # --------------------------------------------------------
    # STANDARD MLP
    # --------------------------------------------------------

    print("\nTraining STANDARD MLP...")

    standard_model, device = train_mlp(
        X_train_processed,
        y_train,
        seed
    )


    # --------------------------------------------------------
    # TTL-AUGMENTED TRAINING DATA
    # --------------------------------------------------------

    print(
        "\nCreating TTL-augmented training data..."
    )

    X_train_augmented_raw = (
        create_ttl_augmented_training_data(
            X_train_raw,
            y_train,
            seed=seed,
            augmentation_strength=AUGMENTATION_STRENGTH,
            fraction=AUGMENT_FRACTION
        )
    )


    n_augmented = int(
        round(
            len(X_train_raw)
            * AUGMENT_FRACTION
        )
    )

    print(
        f"Augmented samples: "
        f"{n_augmented} / {len(X_train_raw)} "
        f"({AUGMENT_FRACTION * 100:.0f}%)"
    )


    # --------------------------------------------------------
    # IMPORTANT:
    # Use the SAME preprocessing pipeline.
    #
    # The preprocessor is fitted ONLY on the original clean
    # training data, then applied to augmented training data.
    # --------------------------------------------------------

    X_train_augmented_processed = (
        preprocessor.transform(
            X_train_augmented_raw
        )
    )

    X_train_augmented_processed = np.asarray(
        X_train_augmented_processed,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # TRAIN TTL-AUGMENTED MLP
    # --------------------------------------------------------

    print(
        "Training TTL-AUGMENTED MLP..."
    )

    augmented_model, augmented_device = train_mlp(
        X_train_augmented_processed,
        y_train,
        seed
    )


    # ========================================================
    # TEST AT EACH dTTL LEVEL
    # ========================================================

    for dttl_level in DTTL_LEVELS:

        # ----------------------------------------------------
        # Create perturbed TEST SET
        # ----------------------------------------------------

        X_test_perturbed_raw = (
            perturb_dttl_test(
                X_test_raw,
                dttl_level
            )
        )

        X_test_perturbed_processed = (
            preprocessor.transform(
                X_test_perturbed_raw
            )
        )

        X_test_perturbed_processed = np.asarray(
            X_test_perturbed_processed,
            dtype=np.float32
        )


        # ----------------------------------------------------
        # STANDARD MODEL
        # ----------------------------------------------------

        standard_prob = predict_mlp(
            standard_model,
            device,
            X_test_perturbed_processed
        )

        (
            standard_tpr,
            standard_auroc,
            standard_auprc
        ) = calculate_metrics(
            y_test,
            standard_prob
        )


        # ----------------------------------------------------
        # TTL-AUGMENTED MODEL
        # ----------------------------------------------------

        augmented_prob = predict_mlp(
            augmented_model,
            augmented_device,
            X_test_perturbed_processed
        )

        (
            augmented_tpr,
            augmented_auroc,
            augmented_auprc
        ) = calculate_metrics(
            y_test,
            augmented_prob
        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        standard_records.append({

            "Seed": seed,
            "dTTL_%": dttl_level,

            "TPR": standard_tpr,
            "AUROC": standard_auroc,
            "AUPRC": standard_auprc
        })


        augmented_records.append({

            "Seed": seed,
            "dTTL_%": dttl_level,

            "TPR": augmented_tpr,
            "AUROC": augmented_auroc,
            "AUPRC": augmented_auprc
        })


# ============================================================
# 15. DATAFRAMES
# ============================================================

standard_df = pd.DataFrame(
    standard_records
)

augmented_df = pd.DataFrame(
    augmented_records
)


# ============================================================
# 16. MEAN ± SD ACROSS THREE SEEDS
# ============================================================

standard_summary = (
    standard_df
    .groupby("dTTL_%")
    .agg(
        TPR_mean=("TPR", "mean"),
        TPR_SD=("TPR", "std"),

        AUROC_mean=("AUROC", "mean"),
        AUROC_SD=("AUROC", "std"),

        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_SD=("AUPRC", "std")
    )
    .reset_index()
)


augmented_summary = (
    augmented_df
    .groupby("dTTL_%")
    .agg(
        TPR_mean=("TPR", "mean"),
        TPR_SD=("TPR", "std"),

        AUROC_mean=("AUROC", "mean"),
        AUROC_SD=("AUROC", "std"),

        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_SD=("AUPRC", "std")
    )
    .reset_index()
)


# ============================================================
# 17. OWN CLEAN BASELINES
# ============================================================

standard_clean_tpr = (
    standard_summary
    .loc[
        standard_summary["dTTL_%"] == 0,
        "TPR_mean"
    ]
    .iloc[0]
)

augmented_clean_tpr = (
    augmented_summary
    .loc[
        augmented_summary["dTTL_%"] == 0,
        "TPR_mean"
    ]
    .iloc[0]
)


# ============================================================
# 18. TPR DEGRADATION FROM EACH MODEL'S OWN CLEAN BASELINE
# ============================================================

standard_summary[
    "TPR_degradation"
] = (
    standard_clean_tpr
    - standard_summary["TPR_mean"]
)


augmented_summary[
    "TPR_degradation"
] = (
    augmented_clean_tpr
    - augmented_summary["TPR_mean"]
)


# ============================================================
# 19. FINAL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame({

    "dTTL_%":
        standard_summary["dTTL_%"],

    "Standard_MLP_TPR":
        standard_summary["TPR_mean"],

    "Standard_MLP_TPR_SD":
        standard_summary["TPR_SD"],

    "TTL_Augmented_MLP_TPR":
        augmented_summary["TPR_mean"],

    "TTL_Augmented_MLP_TPR_SD":
        augmented_summary["TPR_SD"],

    "Standard_AUROC":
        standard_summary["AUROC_mean"],

    "Standard_AUROC_SD":
        standard_summary["AUROC_SD"],

    "Augmented_AUROC":
        augmented_summary["AUROC_mean"],

    "Augmented_AUROC_SD":
        augmented_summary["AUROC_SD"],

    "Standard_TPR_Degradation":
        standard_summary["TPR_degradation"],

    "Augmented_TPR_Degradation":
        augmented_summary["TPR_degradation"]
})


# ============================================================
# 20. PRINT CLEAN BASELINES
# ============================================================

print("\n" + "=" * 100)
print("CLEAN BASELINES")
print("=" * 100)

print(
    f"Standard MLP clean TPR : "
    f"{standard_clean_tpr:.6f}"
)

print(
    f"TTL-Augmented clean TPR: "
    f"{augmented_clean_tpr:.6f}"
)


# ============================================================
# 21. PRINT PAPER-READY TABLE
# ============================================================

print("\n" + "=" * 100)
print("TABLE 8 — STANDARD AND TTL-AUGMENTED MLP")
print("UNDER dTTL PERTURBATIONS")
print("=" * 100)

print(
    "dTTL   "
    "Standard MLP TPR      "
    "TTL-Augmented MLP TPR      "
    "Standard AUROC      "
    "Augmented AUROC"
)

print("-" * 100)

for _, row in comparison.iterrows():

    print(
        f"{row['dTTL_%']:>+4.0f}%   "

        f"{row['Standard_MLP_TPR']:.4f} ± "
        f"{row['Standard_MLP_TPR_SD']:.4f}      "

        f"{row['TTL_Augmented_MLP_TPR']:.4f} ± "
        f"{row['TTL_Augmented_MLP_TPR_SD']:.4f}      "

        f"{row['Standard_AUROC']:.4f} ± "
        f"{row['Standard_AUROC_SD']:.4f}      "

        f"{row['Augmented_AUROC']:.4f} ± "
        f"{row['Augmented_AUROC_SD']:.4f}"
    )


# ============================================================
# 22. TPR DEGRADATION TABLE
# ============================================================

print("\n" + "=" * 100)
print("TPR DEGRADATION FROM EACH MODEL'S OWN CLEAN BASELINE")
print("=" * 100)

print(
    comparison[
        [
            "dTTL_%",
            "Standard_MLP_TPR",
            "Standard_TPR_Degradation",
            "TTL_Augmented_MLP_TPR",
            "Augmented_TPR_Degradation"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 23. SAVE RESULTS
# ============================================================

standard_df.to_csv(
    "standard_mlp_dttl_results_50pct.csv",
    index=False
)

augmented_df.to_csv(
    "ttl_augmented_mlp_dttl_results_50pct.csv",
    index=False
)

comparison.to_csv(
    "table8_standard_vs_ttl_augmented_50pct.csv",
    index=False
)


print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)

print(
    "standard_mlp_dttl_results_50pct.csv"
)

print(
    "ttl_augmented_mlp_dttl_results_50pct.csv"
)

print(
    "table8_standard_vs_ttl_augmented_50pct.csv"
)

print("\nExperiment completed.")

TTL AUGMENTATION ROBUSTNESS EXPERIMENT
Training samples : 175341
Testing samples  : 82332
Features         : 43

Augmentation fraction: 0.5
Augmentation strength: ±15%
Seeds: [42, 123, 2026]

Numeric features    : 40
Categorical features: 3

Fitting preprocessing pipeline...
Processed train shape: (175341, 195)
Processed test shape : (82332, 195)

SEED 42

Training STANDARD MLP...

Creating TTL-augmented training data...
Augmented samples: 87670 / 175341 (50%)


/tmp/ipykernel_58/801009625.py:321: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[228.16014848 246.83777428 235.76118724 ... 245.9947687  238.30392167
  31.05251225]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[
/tmp/ipykernel_58/801009625.py:332: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[259.64933784   0.           0.         ... 224.252581     0.
  25.91669272]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[


Training TTL-AUGMENTED MLP...

SEED 123

Training STANDARD MLP...

Creating TTL-augmented training data...
Augmented samples: 87670 / 175341 (50%)


/tmp/ipykernel_58/801009625.py:321: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 30.99159065 278.45649673 231.11655421 ...  66.93157966 280.4013872
  32.92673126]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[
/tmp/ipykernel_58/801009625.py:332: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 27.48766992 282.16988481   0.         ... 214.5176194    0.
  27.42981689]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[


Training TTL-AUGMENTED MLP...

SEED 2026

Training STANDARD MLP...

Creating TTL-augmented training data...
Augmented samples: 87670 / 175341 (50%)


/tmp/ipykernel_58/801009625.py:321: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[234.04427719 242.25977351 234.14508902 ...  28.66136396  26.63566343
 279.09214699]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[
/tmp/ipykernel_58/801009625.py:332: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[  0.         274.7127475  250.7663343  ...  28.43497228  25.00212527
   0.        ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_aug.loc[


Training TTL-AUGMENTED MLP...

CLEAN BASELINES
Standard MLP clean TPR : 0.423520
TTL-Augmented clean TPR: 0.423696

TABLE 8 — STANDARD AND TTL-AUGMENTED MLP
UNDER dTTL PERTURBATIONS
dTTL   Standard MLP TPR      TTL-Augmented MLP TPR      Standard AUROC      Augmented AUROC
----------------------------------------------------------------------------------------------------
 -15%   0.3853 ± 0.0157      0.4176 ± 0.0129      0.6467 ± 0.0121      0.5930 ± 0.0227
 -10%   0.3962 ± 0.0164      0.4199 ± 0.0131      0.6513 ± 0.0123      0.5940 ± 0.0227
  -5%   0.4091 ± 0.0172      0.4217 ± 0.0128      0.6562 ± 0.0125      0.5949 ± 0.0228
  -2%   0.4174 ± 0.0175      0.4229 ± 0.0128      0.6593 ± 0.0126      0.5955 ± 0.0228
  +0%   0.4235 ± 0.0175      0.4237 ± 0.0128      0.6615 ± 0.0127      0.5959 ± 0.0229
  +2%   0.4290 ± 0.0173      0.4243 ± 0.0129      0.6636 ± 0.0129      0.5962 ± 0.0229
  +5%   0.4373 ± 0.0164      0.4254 ± 0.0129      0.6667 ± 0.0132      0.5968 ± 0.0230
 +10%   0.4501 ±

In [35]:
# ============================================================
# REPRODUCIBLE MLP AUGMENTATION EXPERIMENT
# UNSW-NB15
#
# 0%       = STANDARD MLP (NO AUGMENTATION)
# ±2%      = TTL augmentation
# ±5%      = TTL augmentation
# ±10%     = TTL augmentation
# ±15%     = TTL augmentation
#
# Seeds: 42, 123, 2026
# ============================================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEEDS = [42, 123, 2026]

AUGMENTATION_LEVELS = [
    0,
    2,
    5,
    10,
    15
]

TARGET = "label"

DROP_COLS = [
    "label",
    "attack_cat"
]

# TTL features
TTL_FEATURES = [
    "sttl",
    "dttl"
]


# ============================================================
# 2. GLOBAL DETERMINISTIC SETTINGS
# ============================================================

def set_seed(seed):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic CUDA behaviour
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. VERIFY DATA
# ============================================================

print("=" * 80)
print("DATA VERIFICATION")
print("=" * 80)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTarget distribution - TRAIN")
print(train_df[TARGET].value_counts())

print("\nTarget distribution - TEST")
print(test_df[TARGET].value_counts())

print("\nChecking TTL features...")

for feature in TTL_FEATURES:

    if feature in train_df.columns:
        print(f"✓ {feature} found")
    else:
        print(f"⚠ {feature} NOT FOUND")


# ============================================================
# 4. FEATURES
# ============================================================

feature_cols = [
    c for c in train_df.columns
    if c not in DROP_COLS
]

X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

y_train = (
    train_df[TARGET]
    .astype(int)
    .to_numpy()
)

y_test = (
    test_df[TARGET]
    .astype(int)
    .to_numpy()
)


# ============================================================
# 5. IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()

print("\nNumeric features    :", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


# ============================================================
# 6. PREPROCESSOR
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_cols
    )
])


print("\nFitting preprocessing pipeline...")

X_train_processed = preprocessor.fit_transform(
    X_train_raw
)

X_test_processed = preprocessor.transform(
    X_test_raw
)

X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)

print(
    "Processed train:",
    X_train_processed.shape
)

print(
    "Processed test :",
    X_test_processed.shape
)


# ============================================================
# 7. MLP ARCHITECTURE
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x).squeeze(1)


# ============================================================
# 8. TTL AUGMENTATION
# ============================================================

def augment_ttl_features(
    X_original,
    feature_columns,
    augmentation_percent,
    seed
):
    """
    Create augmented training samples by applying
    controlled multiplicative perturbations to TTL features.

    For ±N%:
        lower = feature * (1 - N/100)
        upper = feature * (1 + N/100)

    The original samples are retained and augmented
    samples are appended.

    0% returns the original training data unchanged.
    """

    # --------------------------------------------------------
    # 0% = STANDARD MLP
    # --------------------------------------------------------

    if augmentation_percent == 0:

        return X_original.copy()


    rng = np.random.default_rng(seed)

    X_augmented = X_original.copy()

    # --------------------------------------------------------
    # Find TTL columns
    # --------------------------------------------------------

    ttl_indices = []

    for feature in TTL_FEATURES:

        if feature in feature_columns:

            idx = feature_columns.index(feature)

            ttl_indices.append(idx)


    if len(ttl_indices) == 0:

        raise ValueError(
            "Neither sttl nor dttl was found "
            "in the feature columns."
        )


    # --------------------------------------------------------
    # Create one perturbed copy
    # --------------------------------------------------------

    X_new = X_original.copy()

    low = 1.0 - augmentation_percent / 100.0
    high = 1.0 + augmentation_percent / 100.0

    for idx in ttl_indices:

        # Controlled random multiplicative perturbation
        scale = rng.uniform(
            low,
            high,
            size=X_original.shape[0]
        ).astype(np.float32)

        X_new[:, idx] = (
            X_original[:, idx] *
            scale
        )


    # --------------------------------------------------------
    # Append augmented data
    # --------------------------------------------------------

    X_augmented = np.vstack([
        X_original,
        X_new
    ])


    return X_augmented


# ============================================================
# 9. TRAIN MLP
# ============================================================

def train_mlp(
    X_train,
    y_train,
    seed,
    epochs=10,
    batch_size=512,
    learning_rate=1e-3
):

    # Set seed BEFORE model creation
    set_seed(seed)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    # --------------------------------------------------------
    # Tensors
    # --------------------------------------------------------

    X_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_train,
        dtype=torch.float32
    )


    dataset = TensorDataset(
        X_tensor,
        y_tensor
    )


    # --------------------------------------------------------
    # Deterministic DataLoader
    # --------------------------------------------------------

    generator = torch.Generator()

    generator.manual_seed(seed)


    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = MLP(
        input_dim=X_train.shape[1]
    ).to(device)


    criterion = nn.BCEWithLogitsLoss()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for batch_X, batch_y in loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                * len(batch_X)
            )


    return model, device


# ============================================================
# 10. EVALUATE MLP
# ============================================================

def evaluate_mlp(
    model,
    device,
    X_test,
    y_test
):

    model.eval()

    X_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    ).to(device)


    with torch.no_grad():

        logits = model(
            X_tensor
        )

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()


    predictions = (
        probabilities >= 0.5
    ).astype(int)


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    tpr = recall_score(
        y_test,
        predictions,
        zero_division=0
    )


    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()


    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )


    auroc = roc_auc_score(
        y_test,
        probabilities
    )


    auprc = average_precision_score(
        y_test,
        probabilities
    )


    return {
        "TPR": tpr,
        "FPR": fpr,
        "AUROC": auroc,
        "AUPRC": auprc
    }


# ============================================================
# 11. RUN AUGMENTATION EXPERIMENT
# ============================================================

all_results = []


print("\n")
print("=" * 80)
print("MLP AUGMENTATION EXPERIMENT")
print("=" * 80)

print(
    "Seeds:",
    SEEDS
)

print(
    "Augmentation levels:",
    AUGMENTATION_LEVELS
)


for augmentation in AUGMENTATION_LEVELS:

    print("\n")
    print("-" * 80)

    if augmentation == 0:

        print(
            "0% AUGMENTATION = STANDARD MLP"
        )

    else:

        print(
            f"±{augmentation}% TTL AUGMENTATION"
        )

    print("-" * 80)


    for seed in SEEDS:

        print(
            f"\nSeed {seed}"
        )


        # ----------------------------------------------------
        # Create training data
        # ----------------------------------------------------

        X_train_aug = augment_ttl_features(
            X_train_processed,
            feature_cols,
            augmentation,
            seed
        )


        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        if augmentation == 0:

            y_train_aug = y_train.copy()

        else:

            y_train_aug = np.concatenate([
                y_train,
                y_train
            ])


        print(
            "Training samples:",
            len(y_train_aug)
        )


        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model, device = train_mlp(
            X_train_aug,
            y_train_aug,
            seed=seed,
            epochs=10,
            batch_size=512,
            learning_rate=1e-3
        )


        # ----------------------------------------------------
        # Evaluate
        # ----------------------------------------------------

        metrics = evaluate_mlp(
            model,
            device,
            X_test_processed,
            y_test
        )


        print(
            f"TPR   : {metrics['TPR']:.6f}"
        )

        print(
            f"FPR   : {metrics['FPR']:.6f}"
        )

        print(
            f"AUROC : {metrics['AUROC']:.6f}"
        )

        print(
            f"AUPRC : {metrics['AUPRC']:.6f}"
        )


        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        all_results.append({

            "augmentation": augmentation,

            "seed": seed,

            "TPR": metrics["TPR"],

            "FPR": metrics["FPR"],

            "AUROC": metrics["AUROC"],

            "AUPRC": metrics["AUPRC"]
        })


# ============================================================
# 12. RAW RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)


print("\n")
print("=" * 80)
print("PER-SEED RESULTS")
print("=" * 80)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 13. MEAN ± SD
# ============================================================

summary = (
    results_df
    .groupby("augmentation")
    .agg(
        TPR_mean=("TPR", "mean"),
        TPR_SD=("TPR", "std"),

        FPR_mean=("FPR", "mean"),
        FPR_SD=("FPR", "std"),

        AUROC_mean=("AUROC", "mean"),
        AUROC_SD=("AUROC", "std"),

        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_SD=("AUPRC", "std")
    )
    .reset_index()
)


# ============================================================
# 14. FORMAT AUGMENTATION LABEL
# ============================================================

def augmentation_label(x):

    if x == 0:
        return "0% (Standard MLP)"

    return f"±{int(x)}%"


summary.insert(
    0,
    "Augmentation",
    summary["augmentation"].apply(
        augmentation_label
    )
)

summary = summary.drop(
    columns=["augmentation"]
)


# ============================================================
# 15. PRINT FINAL TABLE
# ============================================================

print("\n")
print("=" * 100)
print("TABLE 7 — MLP PERFORMANCE ACROSS AUGMENTATION LEVELS")
print("=" * 100)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 16. PAPER-READY TABLE
# ============================================================

paper_table = summary.copy()

paper_table["TPR"] = (
    paper_table["TPR_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + paper_table["TPR_SD"].map(lambda x: f"{x:.4f}")
)

paper_table["FPR"] = (
    paper_table["FPR_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + paper_table["FPR_SD"].map(lambda x: f"{x:.4f}")
)

paper_table["AUROC"] = (
    paper_table["AUROC_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + paper_table["AUROC_SD"].map(lambda x: f"{x:.4f}")
)

paper_table["AUPRC"] = (
    paper_table["AUPRC_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + paper_table["AUPRC_SD"].map(lambda x: f"{x:.4f}")
)


paper_table = paper_table[
    [
        "Augmentation",
        "TPR",
        "FPR",
        "AUROC",
        "AUPRC"
    ]
]


print("\n")
print("=" * 100)
print("PAPER-READY TABLE 7")
print("=" * 100)

print(
    paper_table.to_string(
        index=False
    )
)


# ============================================================
# 17. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "MLP_augmentation_per_seed_results.csv",
    index=False
)

summary.to_csv(
    "MLP_augmentation_summary.csv",
    index=False
)

paper_table.to_csv(
    "Table_7_MLP_Augmentation.csv",
    index=False
)


print("\nSaved:")
print("✓ MLP_augmentation_per_seed_results.csv")
print("✓ MLP_augmentation_summary.csv")
print("✓ Table_7_MLP_Augmentation.csv")


# ============================================================
# 18. CLEAN STANDARD MLP RESULT
# ============================================================

standard_results = results_df[
    results_df["augmentation"] == 0
].copy()


print("\n")
print("=" * 80)
print("STANDARD MLP — 0% AUGMENTATION")
print("=" * 80)

print(
    standard_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\nSTANDARD MLP MEAN ± SD")

for metric in [
    "TPR",
    "FPR",
    "AUROC",
    "AUPRC"
]:

    mean = standard_results[
        metric
    ].mean()

    sd = standard_results[
        metric
    ].std()

    print(
        f"{metric:6s}: "
        f"{mean:.6f} ± {sd:.6f}"
    )

DATA VERIFICATION
Train shape: (175341, 45)
Test shape : (82332, 45)

Target distribution - TRAIN
label
1    119341
0     56000
Name: count, dtype: int64

Target distribution - TEST
label
1    45332
0    37000
Name: count, dtype: int64

Checking TTL features...
✓ sttl found
✓ dttl found

Numeric features    : 40
Categorical features: 3

Fitting preprocessing pipeline...
Processed train: (175341, 195)
Processed test : (82332, 195)


MLP AUGMENTATION EXPERIMENT
Seeds: [42, 123, 2026]
Augmentation levels: [0, 2, 5, 10, 15]


--------------------------------------------------------------------------------
0% AUGMENTATION = STANDARD MLP
--------------------------------------------------------------------------------

Seed 42
Training samples: 175341
TPR   : 0.437483
FPR   : 0.321865
AUROC : 0.671884
AUPRC : 0.753734

Seed 123
Training samples: 175341
TPR   : 0.403909
FPR   : 0.280838
AUROC : 0.665196
AUPRC : 0.748217

Seed 2026
Training samples: 175341
TPR   : 0.429167
FPR   : 0.250649
AURO

In [36]:
# ============================================================
# TABLE 8
# STANDARD MLP vs TTL-AUGMENTED MLP
# UNDER dTTL PERTURBATIONS
#
# Standard MLP:
#     0% training augmentation
#
# TTL-Augmented MLP:
#     ±AUGMENT_PCT training augmentation
#
# Test perturbations:
#     -15%, -10%, -5%, -2%, 0%, +2%, +5%, +10%, +15%
#
# Seeds:
#     42, 123, 2026
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

SEEDS = [42, 123, 2026]

# ------------------------------------------------------------
# IMPORTANT:
# Set this to the augmentation level used for your
# TTL-Augmented MLP.
#
# Example:
# 15 = ±15% TTL augmentation
# 10 = ±10% TTL augmentation
# 5  = ±5% TTL augmentation
# ------------------------------------------------------------

AUGMENT_PCT = 15


# dTTL test perturbation levels

DTTL_LEVELS = [
    -15,
    -10,
    -5,
    -2,
    0,
    2,
    5,
    10,
    15
]


TARGET = "label"

DROP_COLS = [
    "label",
    "attack_cat"
]

TTL_FEATURES = [
    "sttl",
    "dttl"
]

DTTL_FEATURE = "dttl"


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 4. VERIFY DATA
# ============================================================

print("=" * 80)
print("TABLE 8 — DATA CHECK")
print("=" * 80)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

if DTTL_FEATURE not in train_df.columns:

    raise ValueError(
        f"{DTTL_FEATURE} is not present in train_df."
    )

if DTTL_FEATURE not in test_df.columns:

    raise ValueError(
        f"{DTTL_FEATURE} is not present in test_df."
    )

print("\n✓ dTTL column found")


# ============================================================
# 5. RAW FEATURES
# ============================================================

feature_cols = [
    c for c in train_df.columns
    if c not in DROP_COLS
]


X_train_raw = train_df[
    feature_cols
].copy()


X_test_raw = test_df[
    feature_cols
].copy()


y_train = (
    train_df[TARGET]
    .astype(int)
    .to_numpy()
)


y_test = (
    test_df[TARGET]
    .astype(int)
    .to_numpy()
)


print(
    "\nNumber of features:",
    len(feature_cols)
)


# ============================================================
# 6. IDENTIFY FEATURE TYPES
# ============================================================

categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()


numeric_cols = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()


print(
    "Numeric features    :",
    len(numeric_cols)
)

print(
    "Categorical features:",
    len(categorical_cols)
)


# ============================================================
# 7. PREPROCESSING
#
# IMPORTANT:
# The preprocessor is fitted ONLY on the clean training data.
#
# It is then used for:
#   clean training data
#   augmented training data
#   clean test data
#   perturbed test data
#
# This prevents preprocessing leakage.
# ============================================================

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([

    (
        "num",
        numeric_pipeline,
        numeric_cols
    ),

    (
        "cat",
        categorical_pipeline,
        categorical_cols
    )
])


print("\nFitting preprocessing on CLEAN training data...")


X_train_clean_processed = (
    preprocessor.fit_transform(
        X_train_raw
    )
)


X_test_clean_processed = (
    preprocessor.transform(
        X_test_raw
    )
)


X_train_clean_processed = np.asarray(
    X_train_clean_processed,
    dtype=np.float32
)


X_test_clean_processed = np.asarray(
    X_test_clean_processed,
    dtype=np.float32
)


print(
    "Processed train:",
    X_train_clean_processed.shape
)

print(
    "Processed test :",
    X_test_clean_processed.shape
)


# ============================================================
# 8. MLP ARCHITECTURE
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                1
            )
        )


    def forward(self, x):

        return self.network(
            x
        ).squeeze(1)


# ============================================================
# 9. CREATE TTL-AUGMENTED TRAINING DATA
#
# Augmentation is performed on RAW feature values BEFORE
# preprocessing.
#
# Original samples are retained.
# Augmented samples are appended.
# ============================================================

def create_ttl_augmented_training_data(
    X_raw,
    y,
    augmentation_percent,
    seed
):

    # --------------------------------------------------------
    # 0% = STANDARD MLP
    # --------------------------------------------------------

    if augmentation_percent == 0:

        return (
            X_raw.copy(),
            y.copy()
        )


    X_original = X_raw.copy()


    # --------------------------------------------------------
    # Random generator
    # --------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )


    # --------------------------------------------------------
    # Create augmented copy
    # --------------------------------------------------------

    X_aug = X_original.copy()


    # --------------------------------------------------------
    # Perturb TTL features
    #
    # Each sample receives a random scale between:
    #
    # 1 - p/100
    # and
    # 1 + p/100
    # --------------------------------------------------------

    for feature in TTL_FEATURES:

        if feature not in X_aug.columns:

            continue


        scale = rng.uniform(

            1.0 -
            augmentation_percent / 100.0,

            1.0 +
            augmentation_percent / 100.0,

            size=len(X_aug)
        )


        X_aug[feature] = (
            X_aug[feature].astype(float)
            * scale
        )


    # --------------------------------------------------------
    # Append original + augmented
    # --------------------------------------------------------

    X_combined = pd.concat(
        [
            X_original,
            X_aug
        ],
        axis=0,
        ignore_index=True
    )


    y_combined = np.concatenate(
        [
            y,
            y
        ]
    )


    return (
        X_combined,
        y_combined
    )


# ============================================================
# 10. TRAIN MLP
# ============================================================

def train_mlp(
    X,
    y,
    seed,
    epochs=10,
    batch_size=512,
    learning_rate=1e-3
):

    # --------------------------------------------------------
    # Set seed BEFORE model initialization
    # --------------------------------------------------------

    set_seed(seed)


    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    # --------------------------------------------------------
    # Tensors
    # --------------------------------------------------------

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )


    y_tensor = torch.tensor(
        y,
        dtype=torch.float32
    )


    dataset = TensorDataset(
        X_tensor,
        y_tensor
    )


    # --------------------------------------------------------
    # Deterministic shuffle
    # --------------------------------------------------------

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    loader = DataLoader(

        dataset,

        batch_size=batch_size,

        shuffle=True,

        generator=generator
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = MLP(
        X.shape[1]
    ).to(device)


    criterion = nn.BCEWithLogitsLoss()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()


    for epoch in range(
        epochs
    ):

        for batch_X, batch_y in loader:

            batch_X = batch_X.to(
                device
            )

            batch_y = batch_y.to(
                device
            )


            optimizer.zero_grad()


            logits = model(
                batch_X
            )


            loss = criterion(
                logits,
                batch_y
            )


            loss.backward()


            optimizer.step()


    return (
        model,
        device
    )


# ============================================================
# 11. APPLY dTTL PERTURBATION TO TEST DATA
#
# This modifies ONLY dTTL.
#
# Example:
#
# -15% → dttl × 0.85
# -10% → dttl × 0.90
#  -5% → dttl × 0.95
#   0% → dttl × 1.00
#  +5% → dttl × 1.05
# +15% → dttl × 1.15
# ============================================================

def perturb_dttl(
    X_raw,
    percentage
):

    X = X_raw.copy()


    multiplier = (
        1.0 +
        percentage / 100.0
    )


    X[DTTL_FEATURE] = (
        X[DTTL_FEATURE].astype(float)
        * multiplier
    )


    return X


# ============================================================
# 12. EVALUATE MODEL
# ============================================================

def evaluate_model(
    model,
    device,
    X_test_processed,
    y_test
):

    model.eval()


    X_tensor = torch.tensor(
        X_test_processed,
        dtype=torch.float32
    ).to(device)


    with torch.no_grad():

        logits = model(
            X_tensor
        )


        probabilities = (
            torch.sigmoid(
                logits
            )
            .cpu()
            .numpy()
        )


    predictions = (
        probabilities >= 0.5
    ).astype(int)


    tpr = recall_score(
        y_test,
        predictions,
        zero_division=0
    )


    auroc = roc_auc_score(
        y_test,
        probabilities
    )


    auprc = average_precision_score(
        y_test,
        probabilities
    )


    return {
        "TPR": tpr,
        "AUROC": auroc,
        "AUPRC": auprc
    }


# ============================================================
# 13. RESULTS STORAGE
# ============================================================

all_results = []


# ============================================================
# 14. MAIN EXPERIMENT
# ============================================================

print("\n")
print("=" * 80)
print("TABLE 8 EXPERIMENT")
print("=" * 80)

print(
    f"TTL augmentation level: ±{AUGMENT_PCT}%"
)

print(
    f"Seeds: {SEEDS}"
)

print(
    f"dTTL levels: {DTTL_LEVELS}"
)


for seed in SEEDS:

    print("\n")
    print("#" * 80)

    print(
        f"SEED {seed}"
    )

    print(
        "#" * 80
    )


    # ========================================================
    # A. STANDARD MLP
    #
    # 0% augmentation
    # ========================================================

    print(
        "\nTraining STANDARD MLP..."
    )


    X_standard_raw = X_train_raw.copy()

    y_standard = y_train.copy()


    X_standard_processed = (
        preprocessor.transform(
            X_standard_raw
        )
    )


    X_standard_processed = np.asarray(
        X_standard_processed,
        dtype=np.float32
    )


    standard_model, standard_device = (
        train_mlp(
            X_standard_processed,
            y_standard,
            seed=seed
        )
    )


    # ========================================================
    # B. TTL-AUGMENTED MLP
    # ========================================================

    print(
        f"Training TTL-AUGMENTED MLP "
        f"(±{AUGMENT_PCT}%)..."
    )


    (
        X_aug_raw,
        y_aug
    ) = create_ttl_augmented_training_data(

        X_train_raw,

        y_train,

        AUGMENT_PCT,

        seed
    )


    # IMPORTANT:
    # Transform using the SAME preprocessor
    # fitted on clean training data.

    X_aug_processed = (
        preprocessor.transform(
            X_aug_raw
        )
    )


    X_aug_processed = np.asarray(
        X_aug_processed,
        dtype=np.float32
    )


    augmented_model, augmented_device = (
        train_mlp(

            X_aug_processed,

            y_aug,

            seed=seed
        )
    )


    # ========================================================
    # C. EVALUATE BOTH MODELS UNDER EVERY dTTL LEVEL
    # ========================================================

    for dttl_level in DTTL_LEVELS:

        print(
            f"\nSeed {seed} | "
            f"dTTL {dttl_level:+d}%"
        )


        # ----------------------------------------------------
        # Create perturbed RAW test data
        # ----------------------------------------------------

        X_test_perturbed_raw = (
            perturb_dttl(
                X_test_raw,
                dttl_level
            )
        )


        # ----------------------------------------------------
        # Transform using SAME preprocessor
        # ----------------------------------------------------

        X_test_perturbed_processed = (
            preprocessor.transform(
                X_test_perturbed_raw
            )
        )


        X_test_perturbed_processed = (
            np.asarray(
                X_test_perturbed_processed,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # STANDARD MLP
        # ----------------------------------------------------

        standard_metrics = evaluate_model(

            standard_model,

            standard_device,

            X_test_perturbed_processed,

            y_test
        )


        # ----------------------------------------------------
        # TTL-AUGMENTED MLP
        # ----------------------------------------------------

        augmented_metrics = evaluate_model(

            augmented_model,

            augmented_device,

            X_test_perturbed_processed,

            y_test
        )


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        all_results.append({

            "Seed": seed,

            "dTTL_%": dttl_level,

            "Standard_TPR":
                standard_metrics["TPR"],

            "Standard_AUROC":
                standard_metrics["AUROC"],

            "Standard_AUPRC":
                standard_metrics["AUPRC"],

            "Augmented_TPR":
                augmented_metrics["TPR"],

            "Augmented_AUROC":
                augmented_metrics["AUROC"],

            "Augmented_AUPRC":
                augmented_metrics["AUPRC"]

        })


        print(
            f"  Standard  TPR={standard_metrics['TPR']:.6f} "
            f"AUROC={standard_metrics['AUROC']:.6f}"
        )

        print(
            f"  Augmented TPR={augmented_metrics['TPR']:.6f} "
            f"AUROC={augmented_metrics['AUROC']:.6f}"
        )


# ============================================================
# 15. RAW PER-SEED RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)


print("\n")
print("=" * 100)
print("PER-SEED RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 16. MEAN AND SD ACROSS THE THREE SEEDS
# ============================================================

summary = (

    results_df

    .groupby("dTTL_%")

    .agg(

        Standard_TPR_mean=(
            "Standard_TPR",
            "mean"
        ),

        Standard_TPR_SD=(
            "Standard_TPR",
            "std"
        ),

        Augmented_TPR_mean=(
            "Augmented_TPR",
            "mean"
        ),

        Augmented_TPR_SD=(
            "Augmented_TPR",
            "std"
        ),

        Standard_AUROC_mean=(
            "Standard_AUROC",
            "mean"
        ),

        Standard_AUROC_SD=(
            "Standard_AUROC",
            "std"
        ),

        Augmented_AUROC_mean=(
            "Augmented_AUROC",
            "mean"
        ),

        Augmented_AUROC_SD=(
            "Augmented_AUROC",
            "std"
        ),

        Standard_AUPRC_mean=(
            "Standard_AUPRC",
            "mean"
        ),

        Standard_AUPRC_SD=(
            "Standard_AUPRC",
            "std"
        ),

        Augmented_AUPRC_mean=(
            "Augmented_AUPRC",
            "mean"
        ),

        Augmented_AUPRC_SD=(
            "Augmented_AUPRC",
            "std"
        )

    )

    .reset_index()
)


# ============================================================
# 17. PAPER-READY TABLE 8
# ============================================================

table8 = pd.DataFrame({

    "dTTL":

        summary["dTTL_%"].map(
            lambda x: f"{x:+d}%"
        ),


    "Standard MLP TPR":

        summary.apply(

            lambda r:
            f"{r['Standard_TPR_mean']:.4f} ± "
            f"{r['Standard_TPR_SD']:.4f}",

            axis=1
        ),


    "TTL-Augmented MLP TPR":

        summary.apply(

            lambda r:
            f"{r['Augmented_TPR_mean']:.4f} ± "
            f"{r['Augmented_TPR_SD']:.4f}",

            axis=1
        ),


    "Standard AUROC":

        summary.apply(

            lambda r:
            f"{r['Standard_AUROC_mean']:.4f} ± "
            f"{r['Standard_AUROC_SD']:.4f}",

            axis=1
        ),


    "Augmented AUROC":

        summary.apply(

            lambda r:
            f"{r['Augmented_AUROC_mean']:.4f} ± "
            f"{r['Augmented_AUROC_SD']:.4f}",

            axis=1
        )

})


# ============================================================
# 18. PRINT FINAL TABLE
# ============================================================

print("\n")
print("=" * 120)
print("TABLE 8 — STANDARD AND TTL-AUGMENTED MLP")
print("UNDER dTTL PERTURBATIONS")
print("=" * 120)

print(
    table8.to_string(
        index=False
    )
)


# ============================================================
# 19. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "Table8_per_seed_results.csv",
    index=False
)


summary.to_csv(
    "Table8_mean_SD_results.csv",
    index=False
)


table8.to_csv(
    "Table8_paper_ready.csv",
    index=False
)


print("\n")
print("=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    "✓ Table8_per_seed_results.csv"
)

print(
    "✓ Table8_mean_SD_results.csv"
)

print(
    "✓ Table8_paper_ready.csv"
)


# ============================================================
# 20. CHECK THE 0% dTTL STANDARD BASELINE
# ============================================================

baseline_0 = summary[
    summary["dTTL_%"] == 0
]


print("\n")
print("=" * 80)
print("0% dTTL — STANDARD MLP CHECK")
print("=" * 80)


if len(baseline_0) == 1:

    row = baseline_0.iloc[0]

    print(
        f"Standard MLP TPR   : "
        f"{row['Standard_TPR_mean']:.6f} "
        f"± {row['Standard_TPR_SD']:.6f}"
    )

    print(
        f"Standard MLP AUROC : "
        f"{row['Standard_AUROC_mean']:.6f} "
        f"± {row['Standard_AUROC_SD']:.6f}"
    )

    print(
        f"Standard MLP AUPRC : "
        f"{row['Standard_AUPRC_mean']:.6f} "
        f"± {row['Standard_AUPRC_SD']:.6f}"
    )


# ============================================================
# 21. ROBUSTNESS DEGRADATION FROM OWN CLEAN BASELINE
#
# This is useful for the Discussion and for your mentor's
# requirement that robustness must NOT be confused with
# clean predictive performance.
# ============================================================

clean_standard_tpr = (
    summary.loc[
        summary["dTTL_%"] == 0,
        "Standard_TPR_mean"
    ].iloc[0]
)


clean_augmented_tpr = (
    summary.loc[
        summary["dTTL_%"] == 0,
        "Augmented_TPR_mean"
    ].iloc[0]
)


summary["Standard_TPR_Degradation"] = (
    clean_standard_tpr
    - summary["Standard_TPR_mean"]
)


summary["Augmented_TPR_Degradation"] = (
    clean_augmented_tpr
    - summary["Augmented_TPR_mean"]
)


print("\n")
print("=" * 100)
print("TPR DEGRADATION FROM EACH MODEL'S OWN CLEAN BASELINE")
print("=" * 100)

print(
    summary[
        [
            "dTTL_%",
            "Standard_TPR_mean",
            "Standard_TPR_Degradation",
            "Augmented_TPR_mean",
            "Augmented_TPR_Degradation"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 22. FINAL NOTE
# ============================================================

print("\n")
print("=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

print(
    f"Standard MLP = 0% training augmentation"
)

print(
    f"TTL-Augmented MLP = ±{AUGMENT_PCT}% training augmentation"
)

print(
    "All results averaged over seeds:",
    SEEDS
)

print(
    "Test perturbations:",
    DTTL_LEVELS
)

TABLE 8 — DATA CHECK
Train shape: (175341, 45)
Test shape : (82332, 45)

✓ dTTL column found

Number of features: 43
Numeric features    : 40
Categorical features: 3

Fitting preprocessing on CLEAN training data...
Processed train: (175341, 195)
Processed test : (82332, 195)


TABLE 8 EXPERIMENT
TTL augmentation level: ±15%
Seeds: [42, 123, 2026]
dTTL levels: [-15, -10, -5, -2, 0, 2, 5, 10, 15]


################################################################################
SEED 42
################################################################################

Training STANDARD MLP...
Training TTL-AUGMENTED MLP (±15%)...

Seed 42 | dTTL -15%
  Standard  TPR=0.396100 AUROC=0.657524
  Augmented TPR=0.432807 AUROC=0.575890

Seed 42 | dTTL -10%
  Standard  TPR=0.408056 AUROC=0.662115
  Augmented TPR=0.433910 AUROC=0.576569

Seed 42 | dTTL -5%
  Standard  TPR=0.422461 AUROC=0.666933
  Augmented TPR=0.434792 AUROC=0.577199

Seed 42 | dTTL -2%
  Standard  TPR=0.430844 AUROC=0.669888
  Aug